# Bugs — walkthrough

Exercises the C core, the Python wrapper, the interactive controls panel, and the recipe round-trip. Run from the `Bugs/` directory (or with the directory on `sys.path`).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
from python.bugs_py import Bugs, import_run
from python.controls import run_with_controls, available_probes

available_probes()

{'activity': 'Per-genome activity (scrolling hash-colored strip)',
 'q_activity': 'Activity quantile profile (decile strip chart)'}

## 1. Smoke test — init, step, free

Construct a `Bugs` instance, initialize a small grid, seed a uniform food source with a bug density, step it a few times headlessly, then free.

In [2]:
sim = Bugs()
sim.init(64, food_inc=0.05, mutation_rate=0.01)
sim.state(food_source='uniform', food_source_value=1.0, seed_density=0.2)

print('t=0  pop=', sim.get_population(),
      ' food_env=', round(sim.get_food_env(), 1))

for _ in range(50):
    sim.step()

print(f't={sim.get_step()}  pop={sim.get_population()}'
      f'  food_env={sim.get_food_env():.1f}'
      f'  food_bug={sim.get_food_bug():.1f}')

sim.free()

t=0  pop= 810  food_env= 4096.0
t=50  pop=551  food_env=967.2  food_bug=894.9


## 2. Interactive run with probes

Open the SDL2 window with the ipywidgets control panel and both probes enabled. The cell returns immediately — widgets appear below, the SDL2 window opens in a subprocess, and the simulation runs in a background thread.

In [3]:
sim = Bugs()
sim.init(256, food_inc=0.05, mutation_rate=0.01)
sim.state(food_source='uniform', food_source_value=1.0, seed_density=0.2)

run_with_controls(
    sim, cell_px=3,
    probes={'activity': True, 'q_activity': True},
)

<Thread(bugs-sim, started daemon 6321696768)>

While the sim is running, cells below still execute — the Jupyter kernel is free. Call `sim.free()` when you're done to close the SDL window and release shared memory.

In [ ]:
sim.free()

## 3. Passing metaparams as a dict

`sim.init(N, **kwargs)` is the only place metaparams go. The constructor (`Bugs(...)`) does **not** take them. To drive from a dict, use `**` unpacking.

**`N` itself can live in the dict** — it binds to the positional `N` parameter. Just don't pass it *both* positionally and in the dict, or you get `TypeError: got multiple values for argument 'N'`.

In [ ]:
params = dict(
    N                 = 256,
    mutation_rate     = 0.005,
    reproduction_food = 30.0,
    movement_cost     = 0.3,
    eat_amount        = 3.0,
    initial_food      = 15.0,
    food_inc          = 0.08,
    food_threshold    = 0.2,
    gdiff             = 1,
)

sim = Bugs()
sim.init(**params)   # N is in the dict, so no positional arg here

print(sim.params_str())   # a round-trip-safe init(...) call

Live-update a metaparam after init via the `update_*` helpers (direct attribute assignment does not propagate to the C library):

In [ ]:
sim.update_food_inc(0.12)
print('food_inc now', sim.food_inc)
sim.free()

## 4. Image-driven food source

`food_source='brightness'` samples a brightness array: pixels with brightness *below* `food_source_thresh` become food (`F_source = 1`), the rest `F_source = 0`. Below, we build a synthetic pattern (ring of food cells) instead of loading a file.

In [10]:
N = 256
yy, xx = np.mgrid[0:N, 0:N]
r = np.sqrt((xx - N/2)**2 + (yy - N/2)**2)
brightness = np.where((r > N*0.38) & (r < N*0.4), 0.0, 1.0).astype(np.float32)
# Dark ring → food; bright interior/exterior → no food.

sim = Bugs()
sim.init(N, food_inc=0.8, mutation_rate=0.01)
sim.state(food_source='brightness', brightness=brightness,
          food_source_thresh=0.5, seed_density=0.15)

run_with_controls(sim, cell_px=3,
                  probes={'activity': True, 'q_activity': True})

<Thread(bugs-sim, started daemon 12969013248)>

In [21]:
N = 512
yy, xx = np.mgrid[0:N, 0:N]
r = np.sqrt((xx - N/2)**2 + (yy - N/2)**2)
brightness = np.where((r > N*0.38) & (r < N*0.4), 0.0, 1.0).astype(np.float32)
# Dark ring → food; bright interior/exterior → no food.

#square:
yy, xx = np.mgrid[0:N, 0:N]
d = np.maximum(np.abs(xx - N/2), np.abs(yy - N/2))   # half-side distance from center
brightness = np.where((d > N*0.292) & (d < N*0.3), 0.0, 1.0).astype(np.float32)
brightness = np.where((d > N/3) & (d < N/3+1), 0.0, 1.0).astype(np.float32)

sim = Bugs()
sim.init(N, food_inc=0.8, mutation_rate=0.01)
sim.state(food_source='brightness', brightness=brightness,
          food_source_thresh=0.5, seed_density=0.15)

run_with_controls(sim, cell_px=3,
                  probes={'activity': True, 'q_activity': True})

<Thread(bugs-sim, started daemon 6321696768)>

In [ ]:
sim.free()

## 5. Custom food source from a numpy array

`food_source='custom'` accepts an arbitrary `(N,N)` float array in `[0,1]` and uses it directly as `F_source` (no thresholding).

In [ ]:
N = 128
yy, xx = np.mgrid[0:N, 0:N]
# Diagonal gradient: food plentiful at one corner, scarce at the other.
F_src = ((xx + yy) / (2*N - 2)).astype(np.float32)

sim = Bugs()
sim.init(N, food_inc=0.05, mutation_rate=0.01)
sim.state(food_source='custom', brightness=F_src, seed_density=0.15)

run_with_controls(sim, cell_px=3, colormode=2,
                  probes={'q_activity': True})

In [ ]:
sim.free()

## 6. Seeded reproducibility

Call `sim.set_seed(...)` **before** `sim.state(...)` — `seed_with_density` uses the RNG, so the seed has to be in place when it runs. Two identical-seed runs should produce identical trajectories.

In [ ]:
def run_and_summarize(seed, steps=100):
    s = Bugs()
    s.init(64, food_inc=0.05, mutation_rate=0.01)
    s.set_seed(seed)
    s.state(food_source='uniform', food_source_value=1.0, seed_density=0.2)
    trajectory = []
    for _ in range(steps):
        s.step()
        trajectory.append((s.get_population(), round(s.get_food_env(), 3)))
    s.free()
    return trajectory

a = run_and_summarize(42)
b = run_and_summarize(42)
c = run_and_summarize(43)

print('seed 42 == seed 42  :', a == b)
print('seed 42 == seed 43  :', a == c)
print('last step of seed 42:', a[-1])
print('last step of seed 43:', c[-1])

## 7. Recipes — export and import

`sim.export_recipe(descriptor, ...)` writes a JSON file under `../Runs/` capturing the current metaparams (both the init-time values and any live-edited "final" values), the food-source configuration, seed density, probes, and color mode.

`import_run(filepath, recipe='init'|'final')` rebuilds a fresh `Bugs` instance from that recipe and returns `(sim, display_kwargs)` — `display_kwargs` is ready to splat into `run_with_controls`.

In [ ]:
sim = Bugs()
sim.init(96, food_inc=0.06, mutation_rate=0.01, reproduction_food=18.0)
sim.set_seed(1234)
sim.state(food_source='uniform', food_source_value=1.0, seed_density=0.25)

recipe_path = sim.export_recipe(
    "walkthrough-demo",
    probes={'activity': True, 'q_activity': True},
    colormode=1,
)
sim.free()
print('wrote', recipe_path)

In [ ]:
# List recipes currently under Runs/
import_run()

In [ ]:
sim, display_kwargs = import_run(recipe_path)
print(sim.params_str())
print('display kwargs:', display_kwargs)

run_with_controls(sim, cell_px=3, **display_kwargs)

In [ ]:
sim.free()

## 8. Programmatic probe access

Probes can be read directly without the display, which is useful for headless experiments. `get_activity()` returns the per-genome hash table as numpy arrays; `q_activity_deciles()` returns the 9 population-activity deciles p10..p90.

In [ ]:
sim = Bugs()
sim.init(64, food_inc=0.08, mutation_rate=0.005)
sim.state(food_source='uniform', food_source_value=1.0, seed_density=0.2)

for _ in range(200):
    sim.step()
    sim.activity_update()

tbl = sim.get_activity()
order = np.argsort(-tbl['activity'])[:10]
print(f"Top 10 genomes by cumulative activity (of {len(tbl['hash'])} distinct):\n")
print(f"  {'hash':>10}  {'activity':>10}  {'pop_count':>10}")
for i in order:
    print(f"  0x{tbl['hash'][i]:08x}  {tbl['activity'][i]:>10}  "
          f"{tbl['pop_count'][i]:>10}")

print('\nq_activity deciles (p10..p90):', sim.q_activity_deciles())
sim.free()